# 🧠 Brain Tumor Segmentation using U-Net — Optimized v3

**Author:** Jauilson Crisostomo da Silva  
**Advisor:** Prof. Dr. Daniel Oliveira Dantas  
**Institution:** Universidade Federal de Sergipe (UFS)  
**Department:** Departamento de Ciências da Computação  
**Dataset:** BraTS 2019 (Brain Tumor Segmentation Challenge)

---

## About this version

This is the **CPU-optimized version** (v3), compatible with **TensorFlow 2.16+ / Keras 3**.

| Aspect | Original | Optimized v3 |
|--------|----------|--------------|
| Target hardware | GPU | CPU / any |
| Model parameters | ~31M | ~7.8M |
| Input resolution | 128×128 | 64×64 |
| Keras import style | `tensorflow.keras` | standalone `keras` |
| Drive retry | No | Yes (3×, 2s delay) |
| Memory management | Standard | float32 + gc.collect() |

**Key bug fixes over original:**
- 4D volume handling: BraTS volumes are `(240, 240, 155, 4)` — original assumed 3D
- `workers` parameter removed from `model.fit()` (not supported in Keras 3 with numpy arrays)
- Imports migrated from `tensorflow.keras` to standalone `keras`


## 0. Hardware Setup & Imports

Configure TensorFlow to use all available CPU cores and set up all imports.
Setting `CUDA_VISIBLE_DEVICES=-1` ensures CPU-only execution even when CUDA is present.


In [ ]:
import os
import time
import multiprocessing

# ── CPU: use all available cores ─────────────────────────────────────────────
NUM_CORES = multiprocessing.cpu_count()
print(f"CPUs available: {NUM_CORES}")

os.environ["TF_NUM_INTEROP_THREADS"] = str(NUM_CORES)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(NUM_CORES)
os.environ["OMP_NUM_THREADS"]        = str(NUM_CORES)
os.environ["TF_CPP_MIN_LOG_LEVEL"]   = "2"   # suppress INFO/WARNING logs
os.environ["CUDA_VISIBLE_DEVICES"]   = "-1"  # CPU-only

import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')  # headless backend — compatible with Colab without display
import matplotlib.pyplot as plt
import gc

from sklearn.model_selection import train_test_split
from scipy.ndimage import zoom
from skimage.transform import resize

# ── TF 2.16+ / Keras 3: import standalone keras, not tensorflow.keras ─────────
import tensorflow as tf
import keras
from keras import layers
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

tf.config.threading.set_inter_op_parallelism_threads(NUM_CORES)
tf.config.threading.set_intra_op_parallelism_threads(NUM_CORES)

print(f"TensorFlow: {tf.__version__} | Keras: {keras.__version__}")
print(f"Devices: {[d.name for d in tf.config.list_physical_devices()]}")


## 1. Mount Google Drive

Mount Drive and verify that the `databrain` folder structure is accessible.

Expected structure:
```
My Drive/
└── databrain/
    ├── imagesTr/   ← BRATS_001.nii.gz ... BRATS_484.nii.gz
    └── labelsTr/   ← BRATS_001.nii.gz ... BRATS_484.nii.gz
```


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# ── Verify folder structure ───────────────────────────────────────────────────
base = "/content/gdrive/My Drive/databrain"
print(f"Drive mounted: {os.path.exists('/content/gdrive')}")
print(f"databrain exists: {os.path.exists(base)}")

for folder in ["imagesTr", "labelsTr", "imagesTs"]:
    path   = os.path.join(base, folder)
    exists = os.path.exists(path)
    print(f"  {folder}: {'✓' if exists else '✗ NOT FOUND'}")
    if exists:
        brats = [f for f in os.listdir(path) if f.startswith("BRATS")]
        print(f"    → {len(brats)} BRATS files found")
        if brats:
            print(f"    → Example: {sorted(brats)[0]}")


## 2. Configuration

Adjust `MAX_SAMPLES` and `IMG_HEIGHT/WIDTH` based on your hardware:

| Setting | Value | When to change |
|---------|-------|----------------|
| `MAX_SAMPLES = 50` | Pilot run | Set to `None` for full 484-sample training |
| `IMG_HEIGHT = 64` | CPU-safe | Set to `128` if GPU available |
| `BATCH_SIZE = 4` | Low RAM | Increase to `8` with GPU |


In [ ]:
class Config:
    """Configuration parameters — optimized for CPU / low RAM."""

    # ── Paths ─────────────────────────────────────────────────────────────────
    DATA_DIR     = "/content/gdrive/My Drive/databrain"
    IMAGES_TRAIN = os.path.join(DATA_DIR, "imagesTr")
    LABELS_TRAIN = os.path.join(DATA_DIR, "labelsTr")
    IMAGES_TEST  = os.path.join(DATA_DIR, "imagesTs")
    RESULTS_DIR  = "/content/results"

    # ── Model ─────────────────────────────────────────────────────────────────
    IMG_HEIGHT   = 64    # use 128 with GPU
    IMG_WIDTH    = 64
    IMG_CHANNELS = 1     # T1 modality (grayscale)
    NUM_CLASSES  = 1     # binary: tumor vs background

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE       = 4     # increase to 8 with GPU
    EPOCHS           = 30
    LEARNING_RATE    = 1e-4
    VALIDATION_SPLIT = 0.2

    # ── Dataset ───────────────────────────────────────────────────────────────
    START_INDEX = 1
    END_INDEX   = 484
    MAX_SAMPLES = 50     # ← set to None for full dataset

    # ── Drive stability ───────────────────────────────────────────────────────
    DRIVE_RETRIES = 3    # retry attempts on connection error
    RETRY_DELAY   = 2    # seconds between retries


config = Config()
os.makedirs(config.RESULTS_DIR, exist_ok=True)

print("Configuration loaded!")
print(f"  Resolution:  {config.IMG_HEIGHT}x{config.IMG_WIDTH}")
print(f"  Max samples: {config.MAX_SAMPLES}")
print(f"  Batch size:  {config.BATCH_SIZE}")
print(f"  Epochs:      {config.EPOCHS}")
print(f"  Results dir: {config.RESULTS_DIR}")


## 3. Data Loading & Preprocessing

### Key design decisions

**4D volume handling:** BraTS 2019 images are 4D `(240, 240, 155, 4)` — four MRI modalities  
(T1, T1-contrast, T2, FLAIR). We select channel 0 (T1) before any other operation.

**Memory efficiency:** volumes are loaded as `float32` (half the RAM of `float64`)  
and deleted immediately after preprocessing.

**Drive retry:** Google Drive FUSE connections can drop transiently.  
The loader retries up to 3 times with a 2-second delay before skipping a file.


In [ ]:
def load_nifti_file(filepath, retries=3, delay=2):
    """Load NIfTI file with automatic retry for Drive instability."""
    for attempt in range(retries):
        try:
            img  = nib.load(filepath)
            data = img.get_fdata(dtype=np.float32)  # float32 = half the RAM of float64
            return data
        except Exception as e:
            if attempt < retries - 1:
                print(f"  [RETRY {attempt+1}/{retries}] {os.path.basename(filepath)}: {e}")
                time.sleep(delay)
            else:
                print(f"  [ERROR] {os.path.basename(filepath)}: {e}")
                return None


def normalize_image(image):
    """Normalize image to [0, 1] range (float32)."""
    img_min, img_max = np.min(image), np.max(image)
    if img_max - img_min > 0:
        return ((image - img_min) / (img_max - img_min)).astype(np.float32)
    return image.astype(np.float32)


def preprocess_volume(volume, target_size=(64, 64)):
    """
    Preprocess a 3D or 4D MRI volume:
      1. 4D → select T1 modality (channel 0)
      2. Normalize intensity to [0, 1]
      3. Extract central axial slice
      4. Resize to target_size
      5. Add channel dimension → (H, W, 1)
    """
    # Step 1: handle 4D multi-modal volumes (BraTS format)
    if len(volume.shape) == 4:
        volume = volume[:, :, :, 0]  # select T1 modality

    # Step 2: normalize
    volume_norm = normalize_image(volume)

    # Step 3: central axial slice
    if len(volume_norm.shape) == 3:
        mid      = volume_norm.shape[2] // 2
        slice_2d = volume_norm[:, :, mid]
    else:
        slice_2d = volume_norm

    # Step 4: resize
    resized = resize(
        slice_2d, target_size,
        preserve_range=True, anti_aliasing=True, order=1
    ).astype(np.float32)

    # Step 5: add channel dimension
    return np.expand_dims(resized, axis=-1)


print("Preprocessing functions defined!")


In [ ]:
def load_dataset(config):
    """
    Load and preprocess the BraTS dataset with memory-efficient handling.

    Returns:
        X: np.array of shape (N, H, W, 1), float32
        y: np.array of shape (N, H, W, 1), float32 — binary tumor mask
    """
    images, labels = [], []

    end_index = min(config.END_INDEX, config.MAX_SAMPLES) \
                if config.MAX_SAMPLES else config.END_INDEX

    total = end_index - config.START_INDEX + 1
    print(f"Loading up to {total} samples | Resolution: {config.IMG_HEIGHT}x{config.IMG_WIDTH}")
    print("This may take a few minutes...\n")

    loaded = skipped = 0

    for i in range(config.START_INDEX, end_index + 1):
        idx        = str(i).zfill(3)
        image_path = os.path.join(config.IMAGES_TRAIN, f"BRATS_{idx}.nii.gz")
        label_path = os.path.join(config.LABELS_TRAIN, f"BRATS_{idx}.nii.gz")

        if not os.path.exists(image_path) or not os.path.exists(label_path):
            skipped += 1
            continue

        image_vol = load_nifti_file(image_path, config.DRIVE_RETRIES, config.RETRY_DELAY)
        label_vol = load_nifti_file(label_path, config.DRIVE_RETRIES, config.RETRY_DELAY)

        if image_vol is None or label_vol is None:
            skipped += 1
            continue

        try:
            img_proc = preprocess_volume(image_vol, (config.IMG_HEIGHT, config.IMG_WIDTH))
            lbl_proc = preprocess_volume(label_vol, (config.IMG_HEIGHT, config.IMG_WIDTH))
            lbl_proc = (lbl_proc > 0).astype(np.float32)  # binarize

            images.append(img_proc)
            labels.append(lbl_proc)
            loaded += 1

            # Free original volumes immediately to save RAM
            del image_vol, label_vol, img_proc, lbl_proc
            if loaded % 10 == 0:
                gc.collect()
                print(f"  ✓ {loaded} images processed...")

        except Exception as e:
            print(f"  [ERROR] BRATS_{idx}: {e}")
            skipped += 1

    print(f"\n✅ Loaded: {loaded} | Skipped: {skipped}")

    if loaded == 0:
        raise RuntimeError(
            "No images loaded! Please verify:\n"
            "  1. drive.mount('/content/gdrive') was executed\n"
            "  2. databrain folder is directly under 'My Drive'\n"
            "  3. Files are named BRATS_001.nii.gz"
        )

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)
    del images, labels
    gc.collect()
    return X, y


print("Dataset loading function defined!")


## 4. Load Dataset

Load and preprocess the BraTS images. Monitor RAM usage before and after.


In [ ]:
def print_memory_usage():
    """Print current RAM usage (requires psutil)."""
    try:
        import psutil
        m = psutil.virtual_memory()
        print(f"  RAM: {m.used/1e9:.1f} GB used / {m.total/1e9:.1f} GB total ({m.percent:.1f}%)")
    except ImportError:
        print("  (psutil not available — install with: pip install psutil)")


print("RAM before loading:")
print_memory_usage()

X, y = load_dataset(config)

print(f"\nDataset shape:")
print(f"  Images: {X.shape} | dtype: {X.dtype}")
print(f"  Labels: {y.shape} | unique values: {np.unique(y)}")
print(f"  Image range: [{X.min():.3f}, {X.max():.3f}]")
print("\nRAM after loading:")
print_memory_usage()


## 5. Data Exploration & Visualization

Visualize sample MRI slices alongside their tumor masks before training.


In [ ]:
def visualize_samples(images, labels, num_samples=5, save_dir=None):
    """Display sample MRI scans and corresponding tumor masks."""
    n   = min(num_samples, len(images))
    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n))

    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        axes[i, 0].imshow(images[i, :, :, 0], cmap='gray')
        axes[i, 0].set_title(f'MRI Scan #{i+1}', fontweight='bold')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(labels[i, :, :, 0], cmap='hot')
        axes[i, 1].set_title(f'Tumor Mask #{i+1}', fontweight='bold')
        axes[i, 1].axis('off')

    plt.suptitle('Sample Data — MRI vs Tumor Mask', fontsize=13)
    plt.tight_layout()

    if save_dir:
        path = os.path.join(save_dir, 'sample_data.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")

    plt.show()


visualize_samples(X, y, num_samples=5, save_dir=config.RESULTS_DIR)


## 6. Train / Validation Split


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=config.VALIDATION_SPLIT,
    random_state=42
)

# Free full arrays — no longer needed
del X, y
gc.collect()

print("Data split completed!")
print(f"  Training samples:   {len(X_train)} → shape {X_train.shape}")
print(f"  Validation samples: {len(X_val)} → shape {X_val.shape}")
print_memory_usage()


## 7. U-Net Model Architecture

**U-Net Light** — filter counts halved vs the original implementation:

| Block | Original filters | Optimized filters |
|-------|-----------------|-------------------|
| Encoder 1 | 64 | 32 |
| Encoder 2 | 128 | 64 |
| Encoder 3 | 256 | 128 |
| Encoder 4 | 512 | 256 |
| Bottleneck | 1024 | 512 |
| **Total params** | **~31M** | **~7.8M** |

Reducing parameters by ~4× results in ~4× faster training on CPU with minimal accuracy impact at low sample counts.


In [ ]:
def conv_block(inputs, num_filters):
    """Double conv block: Conv2D → BatchNorm → Conv2D → BatchNorm."""
    x = layers.Conv2D(num_filters, 3, activation='relu', padding='same',
                      kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(num_filters, 3, activation='relu', padding='same',
                      kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    return x


def encoder_block(inputs, num_filters):
    """Encoder: conv_block + MaxPooling. Returns (skip, pooled)."""
    x = conv_block(inputs, num_filters)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(inputs, skip_features, num_filters):
    """Decoder: Conv2DTranspose + skip connection concatenation + conv_block."""
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    x = layers.concatenate([x, skip_features])
    x = conv_block(x, num_filters)
    return x


def build_unet(input_shape, num_classes=1):
    """
    Build U-Net Light architecture.
    ~7.8M parameters — 4× faster than original (~31M) on CPU.
    """
    inputs = keras.Input(shape=input_shape)

    # Encoder (contracting path)
    s1, p1 = encoder_block(inputs, 32)
    s2, p2 = encoder_block(p1,    64)
    s3, p3 = encoder_block(p2,   128)
    s4, p4 = encoder_block(p3,   256)

    # Bottleneck
    b1 = conv_block(p4, 512)

    # Decoder (expanding path)
    d1 = decoder_block(b1, s4, 256)
    d2 = decoder_block(d1, s3, 128)
    d3 = decoder_block(d2, s2,  64)
    d4 = decoder_block(d3, s1,  32)

    # Output: sigmoid for binary segmentation
    outputs = layers.Conv2D(1, 1, activation='sigmoid', padding='same')(d4)

    return keras.Model(inputs=inputs, outputs=outputs, name='U-Net-Light')


print("Model architecture defined!")


## 8. Evaluation Metrics

- **Dice Coefficient:** measures overlap between prediction and ground truth — standard metric for medical image segmentation
- **IoU (Intersection over Union):** also known as Jaccard index — stricter than Dice
- **Dice Loss:** `1 - Dice` — used as an alternative training loss for class-imbalanced segmentation


In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Dice coefficient: 2 * |A ∩ B| / (|A| + |B|)
    Range: [0, 1] — higher is better.
    """
    y_true_f     = tf.keras.backend.flatten(y_true)
    y_pred_f     = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth
    )


def dice_loss(y_true, y_pred):
    """Dice loss: 1 - Dice coefficient. Use as alternative to binary_crossentropy."""
    return 1 - dice_coefficient(y_true, y_pred)


def iou_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Intersection over Union (Jaccard index): |A ∩ B| / |A ∪ B|
    Range: [0, 1] — higher is better. Stricter than Dice.
    """
    y_true_f     = tf.keras.backend.flatten(y_true)
    y_pred_f     = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union        = (tf.keras.backend.sum(y_true_f)
                    + tf.keras.backend.sum(y_pred_f) - intersection)
    return (intersection + smooth) / (union + smooth)


print("Metrics defined: dice_coefficient, dice_loss, iou_coefficient")


## 9. Model Compilation & Training

**Callbacks used:**
- `ModelCheckpoint` — saves best model by `val_dice_coefficient`
- `EarlyStopping` — stops if `val_loss` doesn't improve for 8 epochs
- `ReduceLROnPlateau` — halves learning rate if `val_loss` stagnates for 4 epochs

> **Note:** `workers` parameter has been removed — not supported in Keras 3 when inputs are numpy arrays.


In [ ]:
input_shape = (config.IMG_HEIGHT, config.IMG_WIDTH, config.IMG_CHANNELS)
model = build_unet(input_shape, num_classes=config.NUM_CLASSES)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss='binary_crossentropy',  # swap for dice_loss if desired
    metrics=['accuracy', dice_coefficient, iou_coefficient]
)

print("=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")


In [ ]:
callbacks = [
    ModelCheckpoint(
        os.path.join(config.RESULTS_DIR, 'best_unet_model.keras'),
        monitor='val_dice_coefficient',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=config.BATCH_SIZE,
    epochs=config.EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "=" * 70)
print("TRAINING COMPLETED!")
print("=" * 70)


## 10. Training History

Plot loss, accuracy, Dice coefficient and IoU across epochs.


In [ ]:
def plot_training_history(history, save_dir=None):
    """Plot training and validation metrics across epochs."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].plot(history.history['loss'],                 label='Train', linewidth=2)
    axes[0,0].plot(history.history['val_loss'],             label='Val',   linewidth=2)
    axes[0,0].set_title('Loss', fontsize=13, fontweight='bold')
    axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
    axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(history.history['accuracy'],             label='Train', linewidth=2)
    axes[0,1].plot(history.history['val_accuracy'],         label='Val',   linewidth=2)
    axes[0,1].set_title('Accuracy', fontsize=13, fontweight='bold')
    axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy')
    axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    axes[1,0].plot(history.history['dice_coefficient'],     label='Train', linewidth=2)
    axes[1,0].plot(history.history['val_dice_coefficient'], label='Val',   linewidth=2)
    axes[1,0].set_title('Dice Coefficient', fontsize=13, fontweight='bold')
    axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Dice')
    axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

    axes[1,1].plot(history.history['iou_coefficient'],      label='Train', linewidth=2)
    axes[1,1].plot(history.history['val_iou_coefficient'],  label='Val',   linewidth=2)
    axes[1,1].set_title('IoU Coefficient', fontsize=13, fontweight='bold')
    axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('IoU')
    axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

    plt.suptitle('Brain Tumor Segmentation — Training History', fontsize=14)
    plt.tight_layout()

    if save_dir:
        path = os.path.join(save_dir, 'training_history.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")
    else:
        plt.show()


plot_training_history(history, save_dir=config.RESULTS_DIR)


## 11. Model Evaluation


In [ ]:
print("=" * 70)
print("MODEL EVALUATION")
print("=" * 70)

results = model.evaluate(X_val, y_val, verbose=1)

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")
print(f"  Validation Loss:             {results[0]:.4f}")
print(f"  Validation Accuracy:         {results[1]:.4f}")
print(f"  Validation Dice Coefficient: {results[2]:.4f}")
print(f"  Validation IoU:              {results[3]:.4f}")
print(f"{'='*70}")


## 12. Segmentation Results Visualization

Side-by-side comparison: original MRI | ground truth mask | U-Net prediction.


In [ ]:
def visualize_predictions(model, X_test, y_test, num_samples=5, save_dir=None):
    """Visualize model predictions vs ground truth."""
    n     = min(num_samples, len(X_test))
    preds = model.predict(X_test[:n], verbose=0)

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        axes[i, 0].imshow(X_test[i, :, :, 0], cmap='gray')
        axes[i, 0].set_title('Original MRI', fontweight='bold')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(y_test[i, :, :, 0], cmap='hot')
        axes[i, 1].set_title('Ground Truth', fontweight='bold')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(preds[i, :, :, 0], cmap='hot')
        axes[i, 2].set_title('U-Net Prediction', fontweight='bold')
        axes[i, 2].axis('off')

    plt.suptitle('Segmentation Results', fontsize=13)
    plt.tight_layout()

    if save_dir:
        path = os.path.join(save_dir, 'segmentation_results.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")
    else:
        plt.show()


visualize_predictions(model, X_val, y_val,
                      num_samples=min(5, len(X_val)),
                      save_dir=config.RESULTS_DIR)


## 13. Save Model & Copy Results to Drive

Model saved in `.keras` format (modern Keras 3 standard).  
Results are also copied to Google Drive for persistence between Colab sessions.


In [ ]:
import shutil

# ── Save model ────────────────────────────────────────────────────────────────
model_path = os.path.join(config.RESULTS_DIR, 'brain_tumor_segmentation_final.keras')
model.save(model_path)
print(f"✅ Model saved: {model_path}")

# ── Copy results to Drive (persists after Colab session ends) ─────────────────
drive_results = os.path.join(config.DATA_DIR, "results")
if os.path.exists("/content/gdrive/My Drive"):
    os.makedirs(drive_results, exist_ok=True)
    shutil.copytree(config.RESULTS_DIR, drive_results, dirs_exist_ok=True)
    print(f"✅ Results copied to Drive: {drive_results}")
else:
    print("⚠️  Drive not mounted — results saved locally only at:", config.RESULTS_DIR)


## 14. Summary & Conclusions

### Pilot Run Results (50 samples, CPU, 14 epochs)

| Metric | Train | Validation |
|--------|-------|------------|
| Loss | 0.6345 | 0.6255 |
| Accuracy | 75.6% | 77.7% |
| Dice Coefficient | 0.1835 | 0.1852 |
| IoU | 0.1015 | 0.1020 |

### Observations

- Validation metrics slightly exceed training metrics in early epochs — the model generalizes well given the small dataset
- Dice of ~0.18 is expected at this scale; literature reports Dice > 0.80 with full dataset + GPU
- EarlyStopping triggered at epoch 14 (best weights from epoch 6)

### Next Steps

1. Set `MAX_SAMPLES = None` and run full 484-sample training with GPU
2. Fix checkerboard artifact — replace `Conv2DTranspose` with `UpSampling2D + Conv2D`
3. Use all 4 MRI modalities (T1, T1c, T2, FLAIR) as multi-channel input
4. Add data augmentation (rotation, flipping, elastic deformation)
5. Implement Attention U-Net for better feature focus
